In [1]:
import os
os.environ["PYARROW_IGNORE_TIMEZONE"] = "1"

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
import pandas as pd

spark = SparkSession.builder \
    .appName("04_Gold_KPIs") \
    .master("local[2]") \
    .config("spark.driver.host", "127.0.0.1") \
    .config("spark.driver.bindAddress", "127.0.0.1") \
    .config("spark.sql.execution.arrow.pyspark.enabled", "true") \
    .config("spark.ui.enabled", "false") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")
print("Spark OK :", spark.version)

Spark OK : 3.5.1


In [2]:
SILVER = "C:/Users/smagu/retail-data-platform/data/silver"

# Lecture des tables Silver via pandas puis conversion Spark
# (evite winutils sur Windows)
clients      = spark.createDataFrame(pd.read_parquet(f"{SILVER}/clients/clients.parquet"))
employes     = spark.createDataFrame(pd.read_parquet(f"{SILVER}/employes/employes.parquet"))
fournisseurs = spark.createDataFrame(pd.read_parquet(f"{SILVER}/fournisseurs/fournisseurs.parquet"))
produits     = spark.createDataFrame(pd.read_parquet(f"{SILVER}/produits/produits.parquet"))
ventes       = spark.createDataFrame(pd.read_parquet(f"{SILVER}/ventes/ventes.parquet"))

print("Tables Silver chargees :")
for nom, df in [("clients", clients), ("employes", employes), ("fournisseurs", fournisseurs),
                ("produits", produits), ("ventes", ventes)]:
    print(f"  {nom} : {df.count()} lignes")

Tables Silver chargees :
  clients : 100 lignes
  employes : 100 lignes
  fournisseurs : 100 lignes
  produits : 100 lignes
  ventes : 86 lignes


In [3]:
# KPI 1 : Chiffre d'affaires par annee et par mois
ca_par_mois = ventes \
    .groupBy("Annee", "Mois") \
    .agg(F.round(F.sum("MontantTotal"), 2).alias("CA_Total"),
         F.count("VenteID").alias("Nb_Ventes")) \
    .orderBy("Annee", "Mois")

print("=== CA PAR ANNEE ET MOIS ===")
ca_par_mois.show(24, truncate=False)

=== CA PAR ANNEE ET MOIS ===
+-----+----+---------+---------+
|Annee|Mois|CA_Total |Nb_Ventes|
+-----+----+---------+---------+
|2020 |1   |1107210.0|2        |
|2020 |2   |2152920.0|2        |
|2020 |3   |154800.0 |1        |
|2020 |5   |2071662.0|3        |
|2020 |6   |335160.0 |1        |
|2020 |7   |2333920.0|2        |
|2020 |8   |4389245.0|5        |
|2020 |9   |1094735.0|2        |
|2020 |10  |244650.0 |1        |
|2020 |11  |875781.0 |1        |
|2020 |12  |3887811.0|2        |
|2021 |1   |8625051.0|2        |
|2021 |2   |6768362.0|4        |
|2021 |3   |1248400.0|2        |
|2021 |4   |7467861.0|4        |
|2021 |5   |2723561.0|2        |
|2021 |6   |753181.0 |1        |
|2021 |7   |2839641.0|2        |
|2021 |8   |2047950.0|1        |
|2021 |9   |379050.0 |1        |
|2021 |10  |503280.0 |1        |
|2021 |11  |299280.0 |1        |
|2021 |12  |9628743.0|4        |
|2022 |1   |1584800.0|2        |
+-----+----+---------+---------+
only showing top 24 rows



In [4]:
# KPI 2 : Top 10 produits par chiffre d'affaires
top_produits = ventes \
    .join(produits, on="ProduitID", how="left") \
    .groupBy("ProduitID", "NomProduit") \
    .agg(F.round(F.sum("MontantTotal"), 2).alias("CA_Total"),
         F.sum("QuantiteVendue").alias("Qte_Vendue"),
         F.count("VenteID").alias("Nb_Ventes")) \
    .orderBy(F.desc("CA_Total")) \
    .limit(10)

print("=== TOP 10 PRODUITS PAR CA ===")
top_produits.show(truncate=False)

=== TOP 10 PRODUITS PAR CA ===
+---------+-----------------------+-----------+----------+---------+
|ProduitID|NomProduit             |CA_Total   |Qte_Vendue|Nb_Ventes|
+---------+-----------------------+-----------+----------+---------+
|87       |Samsung Galaxy S21     |1.9479501E7|19499     |1        |
|65       |Dell XPS 13            |1.7982E7   |18000     |1        |
|46       |Ralph Lauren Polo Shirt|1.1323101E7|16199     |2        |
|70       |Nike Air Max           |8188152.0  |10248     |2        |
|44       |Samsung TV 55'         |6423900.0  |16100     |1        |
|25       |KitchenAid Mixer       |5592000.0  |8000      |2        |
|97       |iPhone 12              |5552442.0  |5558      |2        |
|84       |Ralph Lauren Polo Shirt|5423541.0  |7759      |2        |
|52       |Dell XPS 13            |5368481.0  |6719      |1        |
|8        |Nike Air Max           |5313350.0  |6650      |1        |
+---------+-----------------------+-----------+----------+---------+



In [5]:
# KPI 3 : Top 10 clients par chiffre d'affaires
top_clients = ventes \
    .join(clients, on="ClientID", how="left") \
    .groupBy("ClientID", "Nom", "Prenom") \
    .agg(F.round(F.sum("MontantTotal"), 2).alias("CA_Total"),
         F.count("VenteID").alias("Nb_Achats")) \
    .orderBy(F.desc("CA_Total")) \
    .limit(10)

print("=== TOP 10 CLIENTS PAR CA ===")
top_clients.show(truncate=False)

=== TOP 10 CLIENTS PAR CA ===
+--------+----------+--------+-----------+---------+
|ClientID|Nom       |Prenom  |CA_Total   |Nb_Achats|
+--------+----------+--------+-----------+---------+
|42      |Coleman   |Dennis  |1.9479501E7|1        |
|46      |Richardson|Cheyenne|1.9257E7   |2        |
|37      |Glenn     |Monica  |1.1113401E7|1        |
|41      |Tucker    |Mr.     |8690243.0  |4        |
|18      |Soto      |Holly   |6423900.0  |1        |
|48      |Johnson   |Amy     |5368481.0  |1        |
|65      |Davis     |Drew    |5313350.0  |1        |
|15      |Smith     |Cynthia |4613400.0  |1        |
|98      |Hess      |Ryan    |4234501.0  |2        |
|63      |Potts     |Wesley  |4233901.0  |1        |
+--------+----------+--------+-----------+---------+



In [6]:
# KPI 4 : Performance des employes
perf_employes = ventes \
    .join(employes, on="EmployeID", how="left") \
    .groupBy("EmployeID", "Nom", "Prenom", "Fonction") \
    .agg(F.round(F.sum("MontantTotal"), 2).alias("CA_Genere"),
         F.count("VenteID").alias("Nb_Ventes"),
         F.round(F.avg("MontantTotal"), 2).alias("Panier_Moyen")) \
    .orderBy(F.desc("CA_Genere"))

print("=== PERFORMANCE DES EMPLOYES ===")
perf_employes.show(10, truncate=False)

=== PERFORMANCE DES EMPLOYES ===
+---------+---------+-------+------------------------+-----------+---------+------------+
|EmployeID|Nom      |Prenom |Fonction                |CA_Genere  |Nb_Ventes|Panier_Moyen|
+---------+---------+-------+------------------------+-----------+---------+------------+
|65       |Watson   |Lance  |Graphic Designer        |2.2160537E7|3        |7386845.67  |
|53       |Peterson |Richard|Administrative Assistant|1.926869E7 |6        |3211448.33  |
|36       |Lucero   |Lindsey|Project Manager         |1.7982E7   |1        |1.7982E7    |
|46       |Reynolds |Melissa|Software Developer      |6423900.0  |1        |6423900.0   |
|19       |Juarez   |Chad   |Sales Representative    |5692400.0  |2        |2846200.0   |
|84       |Adams    |Kenneth|Graphic Designer        |5368481.0  |1        |5368481.0   |
|64       |Rogers   |Hector |Administrative Assistant|5109102.0  |2        |2554551.0   |
|22       |Lee      |Gary   |Data Analyst            |4763000.0  |2

In [7]:
# KPI 5 : CA par fournisseur
ca_fournisseurs = ventes \
    .join(produits, on="ProduitID", how="left") \
    .join(fournisseurs, on="FournisseurID", how="left") \
    .groupBy("FournisseurID", "NomFournisseur") \
    .agg(F.round(F.sum("MontantTotal"), 2).alias("CA_Total"),
         F.count("VenteID").alias("Nb_Ventes"),
         F.countDistinct("ProduitID").alias("Nb_Produits")) \
    .orderBy(F.desc("CA_Total")) \
    .limit(10)

print("=== TOP 10 FOURNISSEURS PAR CA ===")
ca_fournisseurs.show(truncate=False)

=== TOP 10 FOURNISSEURS PAR CA ===
+-------------+--------------------------+-----------+---------+-----------+
|FournisseurID|NomFournisseur            |CA_Total   |Nb_Ventes|Nb_Produits|
+-------------+--------------------------+-----------+---------+-----------+
|8            |Brown-Walker              |2.2597673E7|5        |3          |
|15           |Russell, Torres and Smith |2.0261416E7|2        |2          |
|84           |Velasquez-Irwin           |1.7457543E7|10       |5          |
|86           |Bridges Ltd               |1.7359923E7|7        |3          |
|40           |Shelton, Robinson and Lara|8942471.0  |3        |3          |
|35           |Miller-Daniel             |8402301.0  |3        |2          |
|98           |Perez-Brown               |8188152.0  |2        |1          |
|91           |Gaines and Sons           |6793802.0  |3        |2          |
|90           |Montgomery, Reed and Marsh|5313350.0  |1        |1          |
|47           |Barnett-Davidson          

In [8]:
# Ecriture Gold en Parquet (via pandas/pyarrow, sans winutils)
GOLD = "C:/Users/smagu/retail-data-platform/data/gold"

gold_tables = {
    "ca_par_mois":     ca_par_mois,
    "top_produits":    top_produits,
    "top_clients":     top_clients,
    "perf_employes":   perf_employes,
    "ca_fournisseurs": ca_fournisseurs,
}

for nom, df in gold_tables.items():
    path = f"{GOLD}/{nom}"
    os.makedirs(path, exist_ok=True)
    df.toPandas().to_parquet(f"{path}/{nom}.parquet", index=False)
    print(f"Ecrit : {path}/{nom}.parquet")

print("\nToutes les tables Gold sont enregistrees !")

Ecrit : C:/Users/smagu/retail-data-platform/data/gold/ca_par_mois/ca_par_mois.parquet
Ecrit : C:/Users/smagu/retail-data-platform/data/gold/top_produits/top_produits.parquet
Ecrit : C:/Users/smagu/retail-data-platform/data/gold/top_clients/top_clients.parquet
Ecrit : C:/Users/smagu/retail-data-platform/data/gold/perf_employes/perf_employes.parquet
Ecrit : C:/Users/smagu/retail-data-platform/data/gold/ca_fournisseurs/ca_fournisseurs.parquet

Toutes les tables Gold sont enregistrees !


In [9]:
# Verification finale
print("=== VERIFICATION GOLD ===")
for nom in gold_tables.keys():
    path = f"{GOLD}/{nom}/{nom}.parquet"
    df_check = pd.read_parquet(path)
    print(f"\n{nom.upper()} — {len(df_check)} lignes, colonnes : {list(df_check.columns)}")

print("\nNotebook 04 termine !")

=== VERIFICATION GOLD ===

CA_PAR_MOIS — 42 lignes, colonnes : ['Annee', 'Mois', 'CA_Total', 'Nb_Ventes']

TOP_PRODUITS — 10 lignes, colonnes : ['ProduitID', 'NomProduit', 'CA_Total', 'Qte_Vendue', 'Nb_Ventes']

TOP_CLIENTS — 10 lignes, colonnes : ['ClientID', 'Nom', 'Prenom', 'CA_Total', 'Nb_Achats']

PERF_EMPLOYES — 60 lignes, colonnes : ['EmployeID', 'Nom', 'Prenom', 'Fonction', 'CA_Genere', 'Nb_Ventes', 'Panier_Moyen']

CA_FOURNISSEURS — 10 lignes, colonnes : ['FournisseurID', 'NomFournisseur', 'CA_Total', 'Nb_Ventes', 'Nb_Produits']

Notebook 04 termine !
